# ワークフローの基本機能

このノートブックでは、フレキシブルワークフローの基本構成を学びます。

フレキシブルワークフローでは、次のように、役割の異なるノードを組み合わせたワークフローを構成します。

- タスクノード：ユーザーと会話しながら必要な情報を聞き出して、情報が揃ったら次のノードに進む
- エージェントノード：ユーザーとの会話は行わず、インストラクションで指示された処理を1回だけ行う
- 関数ノード：通常の関数で直前のノードの出力結果を受け取り、必要な処理を行う
- 分岐処理：特定のノードの処理結果に応じて、次に進むノードを決定する

## 事前準備

**[WBF-01]**

ADK と Gemini API の利用に必要なパッケージをインストールします。

ADK は開発の速度が速く、後方互換性のない機能変更が行われることがあります。ここでは、安全のために動作確認ができたバーションを指定してインストールしています。

**最後に `RESTART SESSION` ボタンが表示された場合は、これをクリックしてセッションを再起動してください。**

**注意**: `ERROR: pip's dependency resolver does not currently take into account...` のようなエラーメッセージが表示される場合がありますが、これは無視して構いません。

In [ ]:
%pip install --upgrade --user \
    google-adk==2.8.0 \
    google-cloud-aiplatform==2.0.1 \
    google-genai==2.20.0

**[WBF-02]**

インストールされたパッケージのバージョンを確認します。

In [3]:
!pip list | grep -E "(google-adk|google-genai|google-cloud-aiplatform)"

google-adk                            2.8.0
google-cloud-aiplatform               2.0.1
google-genai                          2.20.0


下記の内容が出力されたことを確認してください。

```
google-adk                               2.8.0
google-cloud-aiplatform                  2.0.1
google-genai                             2.20.0
```

## ユーザー認証

**[WBF-03]**

変数 `PROJECT_ID` に事前に準備したプロジェクトのプロジェクト ID を指定してください。

**注意**: プロジェクトを作成したユーザーアカウントと Colab を使用中のユーザーアカウントが一致している必要があります。


In [ ]:
###
PROJECT_ID = 'Project ID を入力'
###

if (not PROJECT_ID) or PROJECT_ID == 'Project ID を入力':
    print('Gemini API を使用する Google Cloud Project の Project ID を入力してください。')
else:
    print(f'変数 PROJECT_ID を設定しました。')
    print(f'PROJECT_ID = {PROJECT_ID}')

**[WBF-04]**

Google Cloud のプロジェクトを使用するためのユーザー認証を行います。

ポップアップ画面の指示に従って、認証処理を行ってください。

許可する操作を選択するチェックボックスが表示された場合は、すべてにチェックを入れてください。

In [3]:
from google.colab import auth
auth.authenticate_user(project_id=PROJECT_ID)

**[WBF-05]**

この後の作業に必要なモジュールをインポートして、LlmAgent オブジェクトが参照する環境変数を設定します。

In [4]:
import os
from IPython.display import Markdown, display
from pydantic import BaseModel, Field
import agentplatform
from agentplatform.frameworks import AdkApp
from google.adk import Event, Workflow
from google.adk.agents.llm_agent import LlmAgent
from google.adk.workflow import DEFAULT_ROUTE

agentplatform.init(project=PROJECT_ID, location='us-central1')
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'True'

**[WBF-06]**

AdkApp オブジェクトとの会話を行う簡易的なアプリケーションのクラス ChatClient を定義します。

ワークフローの進捗にあわせて結果を表示する `async_output` オプションを追加しています。

In [5]:
class ChatClient:
    def __init__(self, adk_app, user_id='default_user'):
        self.adk_app = adk_app
        self.user_id = user_id
        self.session_id = None

    async def async_stream_query(self, message, async_output=False):
        if not self.session_id:
            session = await self.adk_app.async_create_session(
                user_id=self.user_id,
            )
            self.session_id = session['id']

        result = []
        async for event in self.adk_app.async_stream_query(
            user_id=self.user_id,
            session_id=self.session_id,
            message=message,
        ):
            if ('content' in event and 'parts' in event['content']):
                response = '\n'.join(
                    [p['text'] for p in event['content']['parts'] if 'text' in p]
                )
                if response:
                    result.append(response)
                    if async_output:
                        display(Markdown(f'**[{event["author"]}]**'))
                        display(Markdown(response))
        if not async_output:
            return '\n'.join(result)

## タスクノードとエージェントノードの定義

**[WBF-07]**

ユーザーの情報（名前と趣味）を収集するタスクノードを定義します。

In [25]:
class UserInformation(BaseModel):
    """ユーザーの情報"""
    name: str = Field(description='ユーザーの名前')
    hobby: str = Field(description='ユーザーの趣味')

instruction = '''
# タスク
1. ユーザーの名前と趣味を質問します。
2. 得られた情報を UserInformation にセットします。
3. UserInformation の情報が揃ったらタスクを終了します。

# 条件
- フレンドリーに会話してください。
- できるだけ名前と趣味をまとめて聞いてください。
'''

user_information_task = LlmAgent(
    name='user_information_task',
    model='gemini-3.5-flash-lite',
    mode='task',
    output_schema=UserInformation,
    description='ユーザーの情報を集めるタスク',
    instruction=instruction,
)

**[WBF-08]**

タスクノードの実行結果をメッセージとして出力する関数ノードを用意します。

In [26]:
async def output_user_information(node_input: UserInformation):
    message = f'''
```
ユーザー情報を確認しました。
・ 名前: {node_input.name}
・ 趣味: {node_input.hobby}
```
'''
    return Event(
        node_output=node_input,
        message=message,
    )

**[WBF-09]**

ユーザーへのあいさつのメッセージを出力するエージェントノードを定義します。

In [27]:
instruction = '''
ユーザー情報に基づいて、海外の旅行先を提案します。
簡単な一文で出力すること。
'''

travel_guide_agent = LlmAgent(
    name='travel_guide_agent',
    model='gemini-3.5-flash-lite',
    description='海外の旅行先を提案するエージェント',
    instruction=instruction,
    input_schema=UserInformation,
)

## ワークフローグラフの定義とワークフローの実行

**[WBF-10]**

ここまでに用意したノードを繋げたワークフローグラフを定義します。

In [28]:
greeting_workflow = Workflow(
    name='greeting_workflow',
    edges=[
        (
            'START',
            user_information_task, output_user_information,
            travel_guide_agent,
        ),
    ],
)

greeting_workflow_app = AdkApp(
    agent=greeting_workflow,
    app_name='greeting_workflow_app',
)

**[WBF-11]**

最初のメッセージを入力します。

In [31]:
chat_client = ChatClient(greeting_workflow_app)

query = '''
こんにちは。
'''
await chat_client.async_stream_query(query, async_output=True)

**[user_information_task]**

こんにちは！はじめまして😊
あなたのことをもっと知りたいのですが、お名前と趣味を教えていただけますか？

**[WBF-12]**

名前と趣味を聞かれていますが、あえて名前だけを返答します。

In [32]:
query = '''
片桐はいりです。
'''
await chat_client.async_stream_query(query, async_output=True)

**[user_information_task]**

片桐はいりさん、お名前を教えていただきありがとうございます！
ちなみに、何か趣味はありますか？😊

**[WBF-13]**

趣味も教えるように促されるので、追加で趣味を返答します。

集められた情報がメッセージとして表示されて、さらにこれに基づいて、あいさつの文章が生成されます。

In [33]:
query = '''
映画鑑賞と、遺跡巡りもよく行きます。
'''
await chat_client.async_stream_query(query, async_output=True)

**[greeting_workflow]**


```
ユーザー情報を確認しました。
・ 名前: 片桐はいり
・ 趣味: 映画鑑賞、遺跡巡り
```


**[travel_guide_agent]**

映画鑑賞と遺跡巡りがお好きな片桐はいりさんには、古代ローマの歴史を感じながら映画のロケ地巡りも楽しめるイタリア・ローマがおすすめです。

**[WBF-14]**

セッション情報に記録された会話履歴を確認します。

In [34]:
session = await chat_client.adk_app.async_get_session(
    user_id = 'default_user',
    session_id = chat_client.session_id,
)

for i, event in enumerate(session.events):
    print(f'\n=== [Event {i+1}] ===')
    print(f'author: {event.author}')
    print(f'content: {event.content}')


=== [Event 1] ===
author: user
content: parts=[Part(
  text="""
こんにちは。
"""
)] role='user'

=== [Event 2] ===
author: user_information_task
content: parts=[Part(
  text="""こんにちは！はじめまして😊
あなたのことをもっと知りたいのですが、お名前と趣味を教えていただけますか？""",
  thought_signature=b'\x01\x8f=k_!\x15tj\x129\x93^G\\\xb4Z\x02\xa6xi\xd9q\x0b0\x06\xb7W\xf5\xfb\x0biVf\x07@\x05\xb9\x9e\x85\x96\x1b\xe30h\xce\xf05w1\t*\x04[\xf0Ts\xe4N\xb5\x1b5\x11\x89>\xb8\x1a\x00\xea \x8d\xfb\x8d\x05LL\x0e\x0c$\xec\xbd\x05\x95'
)] role='model'

=== [Event 3] ===
author: user
content: parts=[Part(
  text="""
片桐はいりです。
"""
)] role='user'

=== [Event 4] ===
author: user_information_task
content: parts=[Part(
  text="""片桐はいりさん、お名前を教えていただきありがとうございます！
ちなみに、何か趣味はありますか？😊""",
  thought_signature=b'\x01\x8f=k_^\x98)\xf1\x95\xf9\xa9\xa9\x07VO\x8b\t\xe2\x96\x1a\xf3\xf9\xfb\x9d\x9c&\xfa\xf1\x00+\xf9<\xcb\xdb\x01F\x9e\xefj\xb3\x8e\xf1j\x8a\xd3\xf7\xa2J\xef\xfbA\xda\xacW\x91E\x01\xc9\x88o\xc6\xf9\x7f\x88\xa5\xf2d\x9d\xf0i%Bz`\xf0\x1a\x12\xbbX\x96\xa5\xff'
)] 

## 分岐処理の実装

**[WBF-15]**

ワークフローを再実行するか確認するタスクノードと、その後処理をする関数ノードを定義します。

In [37]:
class HumanCheckResult(BaseModel):
    """ワークフロー再実行の判断結果"""
    restart: bool = Field(description='判断結果')

instruction = '''
# タスク
1. ワークフローを再実行するかユーザーに質問します。
2. 得られた情報を HumanCheckResult.restart にセットします。
  - 再実行する場合は True
  - 再実行しない場合は False
3. HumanCheckResult.restart をセットしたらタスクを終了します。

# 条件
- 余計な会話はしないで、「はい」か「いいえ」の判断を求めてください。
'''

human_check_task = LlmAgent(
    name='human_check_task',
    model='gemini-3.5-flash-lite',
    mode='task',
    output_schema=HumanCheckResult,
    description='ワークフロー再実行の判断を受け取るタスク',
    instruction=instruction,
)

def process_human_check_result(node_input: HumanCheckResult):
    if node_input.restart:
        return Event(
            message='ワークフローを再実行します。',
            route='restart',
        )
    else:
        return Event(
            message='ワークフローを終了します。',
            route='end',
        )


**[WBF-16]**

分岐処理を追加したワークフローを定義ます。

分岐処理は、関数ノードが出力した Event オブジェクトの `restart` オプションの値で次のノードを決定します。

In [38]:
async def end_node():
    return Event(message='ワークフローが完了しました。')

greeting_workflow = Workflow(
    name='greeting_workflow',
    edges=[
        (
            'START',
            user_information_task, output_user_information,
            travel_guide_agent,
            human_check_task, process_human_check_result,
        ),
        (
            process_human_check_result,
            {
                'restart': user_information_task,
                DEFAULT_ROUTE: end_node,
            },
        ),
        (
            end_node,
        )
    ],
)

greeting_workflow_app = AdkApp(
    agent=greeting_workflow,
    app_name='greeting_workflow_app',
)

**[WBF-17]**

最初のメッセージを入力します。

In [41]:
chat_client = ChatClient(greeting_workflow_app)

query = '''
こんにちは。
'''
await chat_client.async_stream_query(query, async_output=True)

**[user_information_task]**

こんにちは！お話しできて嬉しいです✨
さっそくですが、あなたの「お名前」と「趣味」を教えていただけますか？

**[WBF-18]**

ユーザーの情報を入力します。

In [42]:
query = '''
片桐はいり。映画鑑賞と遺跡巡り。
'''
await chat_client.async_stream_query(query, async_output=True)

**[greeting_workflow]**


```
ユーザー情報を確認しました。
・ 名前: 片桐はいり
・ 趣味: 映画鑑賞と遺跡巡り
```


**[travel_guide_agent]**

映画のロケ地巡りと古代アステカの遺跡探訪が同時に楽しめる、メキシコシティはいかがでしょうか。

**[human_check_task]**

ワークフローを再実行しますか？「はい」か「いいえ」でお答えください。

**[WBF-19]**

再実行の確認に「はい」で答えます。

In [43]:
query = '''
はい。お願いします。
'''
await chat_client.async_stream_query(query, async_output=True)

**[greeting_workflow]**

ワークフローを再実行します。

**[user_information_task]**

こんにちは！はじめまして😊
あなたについてもっと知りたいのですが、お名前と、何か趣味や好きなことはありますか？教えていただけると嬉しいです！

**[WBF-20]**

ユーザーの情報を入力します。

In [44]:
query = '''
小林聡美。俳句と落語が趣味。
'''
await chat_client.async_stream_query(query, async_output=True)

**[greeting_workflow]**


```
ユーザー情報を確認しました。
・ 名前: 小林聡美
・ 趣味: 俳句と落語
```


**[travel_guide_agent]**

和の趣を感じる京都でのんびりと寺社仏閣巡りや落語を楽しむ旅はいかがでしょうか。

**[human_check_task]**

ワークフローを再実行しますか？「はい」か「いいえ」でお答えください。

**[WBF-21]**

再実行の確認に「中止してください」と答えます。

「はい」「いいえ」以外の表現でも正しく理解されることがわかります。

In [45]:
query = '''
中止してください。
'''
await chat_client.async_stream_query(query, async_output=True)

**[greeting_workflow]**

ワークフローを終了します。

**[greeting_workflow]**

ワークフローが完了しました。